In [20]:
import pandas as pd
from collections import Counter

In [4]:
files = {
    "df2015": "../clean_data/df2015.csv",
    "df2016": "../clean_data/df2016.csv",
    "df2017": "../clean_data/df2017.csv",
    "df2018": "../clean_data/df2018.csv",
    "df2019": "../clean_data/df2019.csv"
}

datasets = {}

for name, path in files.items():
    try:
        df = pd.read_csv(path, sep=",")
        datasets[name] = df
        print(f"{name} cargado correctamente ({len(df)} filas).")
    except Exception as e:
        print(f"Ha habido un error al cargar los datasets: {e}")

df2015 cargado correctamente (158 filas).
df2016 cargado correctamente (157 filas).
df2017 cargado correctamente (155 filas).
df2018 cargado correctamente (156 filas).
df2019 cargado correctamente (156 filas).


Comprobemos los países que coinciden en todos los datasets y los que no

In [10]:
paises = {name: set(df["Country or region"]) for name, df in datasets.items()}
paises_comunes = set.intersection(*paises.values())
len(paises_comunes)

141

In [12]:
paises_totales = set.union(*paises.values())
paises_diferentes = paises_totales - paises_comunes
len(paises_diferentes)

29

In [ ]:
for country in paises_totales:
    presentes = [name for name, val in paises.items() if country in val]
    if len(presentes) != len(datasets):
        print(f"{country}: {presentes}")

Somalia: ['df2016', 'df2017', 'df2018', 'df2019']
North Cyprus: ['df2015', 'df2016', 'df2017']
Swaziland: ['df2015', 'df2019']
Central African Republic: ['df2015', 'df2017', 'df2018', 'df2019']
Angola: ['df2015', 'df2016', 'df2017', 'df2018']
Taiwan: ['df2015', 'df2016', 'df2018', 'df2019']
Oman: ['df2015']
South Sudan: ['df2016', 'df2017', 'df2018', 'df2019']
Comoros: ['df2015', 'df2016', 'df2019']
Suriname: ['df2015', 'df2016']
Belize: ['df2016', 'df2017', 'df2018']
Djibouti: ['df2015']
North Macedonia: ['df2019']
Trinidad and Tobago: ['df2015', 'df2016', 'df2017']
Gambia: ['df2019']
Taiwan Province of China: ['df2017']
Namibia: ['df2016', 'df2017', 'df2018', 'df2019']
Northern Cyprus: ['df2018', 'df2019']
Somaliland region: ['df2015']
Hong Kong S.A.R., China: ['df2017']
Trinidad & Tobago: ['df2018', 'df2019']
Mozambique: ['df2015', 'df2017', 'df2018', 'df2019']
Puerto Rico: ['df2016']
Somaliland Region: ['df2016']
Hong Kong: ['df2015', 'df2016', 'df2018', 'df2019']
Lesotho: ['df2015

In [38]:
contador = Counter()
for s in paises.values():
    contador.update(s)

paises_solo_uno = {country for country, count in contador.items() if count == 1}

for country in paises_solo_uno:
    presente_en = [name for name, ps in paises.items() if country in ps]
    print(country, "-->", presente_en)

Gambia --> ['df2019']
Hong Kong S.A.R., China --> ['df2017']
North Macedonia --> ['df2019']
Taiwan Province of China --> ['df2017']
Puerto Rico --> ['df2016']
Djibouti --> ['df2015']
Somaliland region --> ['df2015']
Oman --> ['df2015']
Somaliland Region --> ['df2016']


Aquellos países que solo aparezcan en un dataset los eliminamos ya que el análisis se centrará en la evolución de los datos en el tiempo

In [ ]:
# Eliminamos los países que solo aparecen en un dataset
for name, df in datasets.items():
    for country in paises_solo_uno:
        datasets[name] = datasets[name][datasets[name]["Country or region"] != country]

In [ ]:
# Comprobamos las nuevas dimensiones de los datasets
for name, df in datasets.items():
    print(datasets[name].shape)

(155, 10)
(155, 10)
(153, 10)
(156, 10)
(154, 10)


Ahora vamos a empezar a comparar la información que nos da cada dataset. Comenzamos mirando el top 5 de cada dataset y el top 5 por abajo para ver cómo ha variado

In [25]:
for name, df in datasets.items():
    print(f"--- Top 5 de {name} ---")
    top5 = df.sort_values(by="Score", ascending=False).head(5)
    print(top5["Country or region"])
    print("\n")


--- Top 5 de df2015 ---
0    Switzerland
1        Iceland
2        Denmark
3         Norway
4         Canada
Name: Country or region, dtype: object


--- Top 5 de df2016 ---
0        Denmark
1    Switzerland
2        Iceland
3         Norway
4        Finland
Name: Country or region, dtype: object


--- Top 5 de df2017 ---
0         Norway
1        Denmark
2        Iceland
3    Switzerland
4        Finland
Name: Country or region, dtype: object


--- Top 5 de df2018 ---
0        Finland
1         Norway
2        Denmark
3        Iceland
4    Switzerland
Name: Country or region, dtype: object


--- Top 5 de df2019 ---
0        Finland
1        Denmark
2         Norway
3        Iceland
4    Netherlands
Name: Country or region, dtype: object




In [26]:
for name, df in datasets.items():
    print(f"--- Top 5 de {name} ---")
    top5 = df.sort_values(by="Score").head(5)
    print(top5["Country or region"])
    print("\n")

--- Top 5 de df2015 ---
157       Togo
156    Burundi
155      Syria
154      Benin
153     Rwanda
Name: Country or region, dtype: object


--- Top 5 de df2016 ---
156        Burundi
155          Syria
154           Togo
153    Afghanistan
152          Benin
Name: Country or region, dtype: object


--- Top 5 de df2017 ---
154    Central African Republic
153                     Burundi
152                    Tanzania
151                       Syria
150                      Rwanda
Name: Country or region, dtype: object


--- Top 5 de df2018 ---
155                     Burundi
154    Central African Republic
153                 South Sudan
152                    Tanzania
151                       Yemen
Name: Country or region, dtype: object


--- Top 5 de df2019 ---
155                 South Sudan
154    Central African Republic
153                 Afghanistan
152                    Tanzania
151                      Rwanda
Name: Country or region, dtype: object




Parece que no cambia mucho la cosa a lo alrgo de los años, vamos a ver qué relaciçon tienen las variables entre sí

In [ ]:
# Vamos a cambiar el tipo del año para que entre dentro de las variables numéricas
for name, df in datasets.items():
    df["year"] = df["year"].astype(int)

In [36]:
datasets["df2015"].info()

<class 'pandas.core.frame.DataFrame'>
Index: 155 entries, 0 to 157
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Overall Rank                  155 non-null    int64  
 1   Country or region             155 non-null    object 
 2   Score                         155 non-null    float64
 3   GDP per Capita                155 non-null    float64
 4   Social Support                155 non-null    float64
 5   Healthy Life Expectancy       155 non-null    float64
 6   Freedom to Make Life Choices  155 non-null    float64
 7   Perceptions of Corruption     155 non-null    float64
 8   Generosity                    155 non-null    float64
 9   year                          155 non-null    int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 13.3+ KB


In [37]:
datasets["df2015"].select_dtypes(include="number").corr()

,Overall Rank,Score,GDP per Capita,Social Support,Healthy Life Expectancy,Freedom to Make Life Choices,Perceptions of Corruption,Generosity,year
Overall Rank,1.000000,-0.992040,-0.785953,-0.732845,-0.734171,-0.556035,-0.384537,-0.165877,NaN
Score,-0.992040,1.000000,0.781479,0.740745,0.722924,0.566785,0.407017,0.187234,NaN
GDP per Capita,-0.785953,0.781479,1.000000,0.647320,0.816294,0.371974,0.340118,0.010302,NaN
Social Support,-0.732845,0.740745,0.647320,1.000000,0.526392,0.447007,0.222867,0.087685,NaN
Healthy Life Expectancy,-0.734171,0.722924,0.816294,0.526392,1.000000,0.363303,0.270849,0.118848,NaN
Freedom to Make Life Choices,-0.556035,0.566785,0.371974,0.447007,0.363303,1.000000,0.491352,0.380625,NaN
Perceptions of Corruption,-0.384537,0.407017,0.340118,0.222867,0.270849,0.491352,1.000000,0.263120,NaN
Generosity,-0.165877,0.187234,0.010302,0.087685,0.118848,0.380625,0.263120,1.000000,NaN
year,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
